# IHDP: fast altitude step response for a quadrotor

In this notebook, an **IHDP (Incremental Heuristic Dynamic Programming)** agent controls the altitude of a nonlinear 6-DoF quadrotor online, with **no feedforward, no PD baseline, and no reference look-ahead**. Thrust is generated by the IHDP neural actor and learned online; transient quality is evaluated with the standard metric set from `tensoraerospace.benchmark`.

## Scenario

* The vehicle starts at an altitude of 1 m (`z₀ = -1 m` in NED).
* At `t = 10 s`, the reference steps to `z = -2 m` (altitude 2 m) and stays there until the end of the episode (`t = 40 s`).
* The first 10 s are a warmup interval at `z = -1 m` for IHDP model identification under a weak 3-2-1-1 PE signal.

## Cascade

* **Inner loop (attitude)** — fixed P controller on `(p, q, r, φ, θ)`.
* **Outer loop (altitude)** — IHDP, which outputs a thrust increment `ΔT` relative to `T_hover = m·g`.

The inner loop stabilizes the attitude so thrust points upward; IHDP handles the altitude task. The actor and critic receive two tracking states: altitude error `e_z = z - z_ref` and vertical body velocity `w_b`, so IHDP can both speed up the transition and damp the motion.

## Tuned Mode

For this task, the best stable configuration uses `Q = diag(300, 10)` and the physically admissible thrust range `T ∈ [0, 30] N`. In one online episode it reaches a `Composite performance index` of about 15, `rise_time ≈ 0.5 s`, `settling_time ≈ 0.6 s`, and nearly zero static error.


**Paper-equation update:** the SISO actor gradient now includes the physical output scale. The actor learning rate and its floor are expressed in these units. Previous cached results were cleared; execute this notebook to obtain results with the current implementation.

## 1. Imports


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from tensoraerospace.aerospacemodel.quadrotor.nonlinear import NonlinearQuadrotor
from tensoraerospace.agent.ihdp.model import IHDPAgent
from tensoraerospace.benchmark.bench import ControlBenchmark

## 2. Simulation Parameters


In [ ]:
DT = 0.01
T_END = 40.0
N = int(T_END / DT) + 1

T_STEP = 10.0          # step time
Z_INIT = -1.0          # initial altitude: 1 m (NED)
Z_FINAL = -2.0         # post-step altitude: 2 m (NED)
STEP_AMPLITUDE = 1.0

# Quadrotor and attitude-controller parameters
M = 1.5
G = 9.81
T_HOVER = M * G        # ~14.71 N, neutral thrust
THRUST_MAX = 30.0      # physical limit of the quadrotor model
MAX_DELTA_T = 12.6     # IHDP authority on ΔT, kept within THRUST_MAX

KP_PQR = 0.05
KP_ANG = 0.30

print(f"Steps: {N}, episode: {T_END} s, dt: {DT} s")
print(f"Initial altitude: {-Z_INIT:.1f} m, target: {-Z_FINAL:.1f} m, step amplitude: {STEP_AMPLITUDE} m")
print(f"Step time: t = {T_STEP} s")
print(f"T_hover = {T_HOVER:.3f} N, ΔT limit = ±{MAX_DELTA_T:.1f} N, T clip = [0, {THRUST_MAX:.1f}] N")

## 3. Reference Signal


In [ ]:
def build_reference(N: int) -> np.ndarray:
    """Altitude reference: step change at T_STEP."""
    ref = np.full(N, Z_INIT, dtype=np.float64)
    ref[int(T_STEP / DT):] = Z_FINAL
    return ref

ref = build_reference(N)
tps = np.arange(N) * DT

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(tps, -ref, color="tab:gray", linestyle="--", label=r"$-z_{ref}$ (reference altitude, m)")
ax.axvline(T_STEP, color="tab:red", linestyle=":", linewidth=0.8, label="step")
ax.set_xlabel("time, s"); ax.set_ylabel("altitude, m")
ax.set_title("Altitude step reference: 1 m → 2 m")
ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()

## 4. IHDP Configuration

The hyperparameters are tuned to minimize `performance_index = 0.4·ISE + 0.4·ITAE + 0.2·|OS|` while keeping a good transient response.

* `tracking_states = ["e_z", "w_b"]` — the actor and critic actually observe altitude error and vertical velocity; `w_b` provides damping without a PD altitude controller.
* `Q_weights = [300, 10]` — the high error weight speeds up the response, while the moderate velocity weight suppresses overshoot.
* Actor `learning_rate = 1.0`, critic `learning_rate = 5.0` — aggressive enough for one online episode.
* `maximum_input = 12.6 N` — IHDP authority on `ΔT`; total thrust is additionally clipped to the physical range `[0, 30] N`.
* `amplitude_3211 = 0.05 N`, `pulse_length_3211 = 10 s` — a weak PE signal active only during the warmup window.
* `NN_initial = 100`, `layers = (12, 1)` — a compact network that converges robustly in this short episode.


In [ ]:
def make_ihdp_agent(N: int) -> IHDPAgent:
    actor_settings = {
        "start_training": 5,
        "layers": (12, 1),
        "activations": ("tanh", "tanh"),
        "learning_rate": 1.0 / MAX_DELTA_T,
    "learning_rate_min": 0.001 / MAX_DELTA_T,
        "learning_rate_exponent_limit": 8,
        "type_PE": "combined",
        "amplitude_3211": 0.05,
        "pulse_length_3211": 10.0 / DT,  # PE decays exactly at the step time
        "maximum_input": MAX_DELTA_T,
        "maximum_q_rate": 30.0,
        "WB_limits": 30,
        "NN_initial": 100,
        "cascade_actor": False,
        "learning_rate_cascaded": 1.2,
    }
    incremental_settings = {
        "number_time_steps": N,
        "dt": DT,
        "input_magnitude_limits": MAX_DELTA_T,
        "input_rate_limits": 500.0,
    }
    critic_settings = {
        "Q_weights": [300.0, 10.0],
        "start_training": -1,
        "gamma": 0.99,
        "learning_rate": 5.0,
        "learning_rate_exponent_limit": 10,
        "layers": (12, 1),
        "activations": ("tanh", "linear"),
        "WB_limits": 30,
        "NN_initial": 100,
        "indices_tracking_states": [0, 1],
    }
    return IHDPAgent(
        actor_settings=actor_settings,
        critic_settings=critic_settings,
        incremental_settings=incremental_settings,
        tracking_states=["e_z", "w_b"],
        selected_states=["e_z", "w_b"],
        selected_input=["dT"],
        number_time_steps=N,
        indices_tracking_states=[0, 1],
    )

## 5. Episode


In [ ]:
def run_episode():
    ref = build_reference(N)
    x0 = np.zeros(12)
    x0[2] = Z_INIT
    model = NonlinearQuadrotor(x0=x0, dt=DT, integrator="rk4")
    agent = make_ihdp_agent(N)

    z_log = np.zeros(N); w_log = np.zeros(N)
    T_log = np.zeros(N); dT_log = np.zeros(N)
    phi_log = np.zeros(N); theta_log = np.zeros(N)

    ref_signal = np.zeros((2, N))

    for k in tqdm(range(N - 1)):
        s = model.current_state
        z_e = s[2]; w_b = s[5]
        phi = s[6]; theta = s[7]
        p, q, r = s[9], s[10], s[11]

        z_log[k] = z_e; w_log[k] = w_b
        phi_log[k] = phi; theta_log[k] = theta

        tau_x = -KP_PQR * p - KP_ANG * phi
        tau_y = -KP_PQR * q - KP_ANG * theta
        tau_z = -KP_PQR * r

        e_z = z_e - ref[k]
        dT_arr = agent.predict(np.array([[e_z], [w_b]]), ref_signal, k)
        dT_cmd = float(np.clip(dT_arr.flatten()[0], -MAX_DELTA_T, MAX_DELTA_T))

        T_total = float(np.clip(T_HOVER + dT_cmd, 0.0, THRUST_MAX))
        dT_actual = T_total - T_HOVER
        T_log[k] = T_total
        dT_log[k] = dT_actual
        model.run_step(np.array([T_total, tau_x, tau_y, tau_z]))

    z_log[-1] = model.current_state[2]
    w_log[-1] = model.current_state[5]
    return ref, z_log, w_log, T_log, dT_log, phi_log, theta_log

ref_log, z_log, w_log, T_log, dT_log, phi_log, theta_log = run_episode()
print("Episode completed.")

## 6. Transient Response Plots


In [ ]:
tps = np.arange(N) * DT
height_actual = -z_log
height_ref = -ref_log
err = z_log - ref_log

band = 0.05 * STEP_AMPLITUDE
band_lo = -Z_FINAL - band
band_hi = -Z_FINAL + band

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(tps, height_ref, "--", color="tab:gray", label=r"$-z_{ref}$")
axes[0].plot(tps, height_actual, color="tab:blue", label=r"$-z$")
axes[0].axhline(band_hi, color="tab:green", linestyle=":", linewidth=0.6, label="±5% settling band")
axes[0].axhline(band_lo, color="tab:green", linestyle=":", linewidth=0.6)
axes[0].axvline(T_STEP, color="tab:red", linestyle=":", linewidth=0.7, label="step")
axes[0].set_ylabel("altitude, m")
axes[0].set_title("IHDP: 1 m → 2 m altitude step response")
axes[0].grid(True); axes[0].legend(loc="lower right")

axes[1].plot(tps[:-1], T_log[:-1], color="tab:red", label=r"$T = T_{hover} + \Delta T$")
axes[1].axhline(T_HOVER, color="tab:gray", linestyle=":", linewidth=0.7, label=r"$T_{hover}$")
axes[1].axvline(T_STEP, color="tab:red", linestyle=":", linewidth=0.7)
axes[1].set_ylabel("thrust, N")
axes[1].grid(True); axes[1].legend(loc="upper right")

axes[2].plot(tps, err, color="tab:purple", label=r"$e_z = z - z_{ref}$")
axes[2].axhline(0, color="tab:gray", linestyle=":", linewidth=0.6)
axes[2].axvline(T_STEP, color="tab:red", linestyle=":", linewidth=0.7)
axes[2].set_xlabel("time, s"); axes[2].set_ylabel("error, m")
axes[2].grid(True); axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 7. Transient Response Metrics (`ControlBenchmark`)


In [ ]:
k_window = int((T_STEP - 1.0) / DT)
control_signal = -ref_log[k_window:]
system_signal = -z_log[k_window:]

bench = ControlBenchmark()
metrics = bench.benchmarking_one_step(
    control_signal=control_signal,
    system_signal=system_signal,
    signal_val=1.0,
    dt=DT,
)

def fmt(v, unit=""):
    if v is None:
        return "undefined"
    if isinstance(v, float):
        return f"{v:.4f}{unit}"
    return f"{v}{unit}"

print(f"=== Transient response metrics (1 m → 2 m step) ===")
print(f"  Overshoot                         : {fmt(metrics['overshoot'], ' %')}")
print(f"  Rise time                         : {fmt(metrics['rise_time'], ' s')}")
print(f"  Settling time                     : {fmt(metrics['settling_time'], ' s')}")
print(f"  Peak time                         : {fmt(metrics['peak_time'], ' s')}")
print(f"  Damping degree                    : {fmt(metrics['damping_degree'])}")
print(f"  Static error                      : {fmt(metrics['static_error'], ' m')}")
print(f"  Maximum deviation                 : {fmt(metrics['maximum_deviation'], ' m')}")
print(f"  Steady-state value                : {fmt(metrics['steady_state_value'], ' m')}")
print(f"  Oscillation count                 : {fmt(metrics['oscillation_count'])}")
print(f"  IAE (∫|e| dt)                     : {fmt(metrics['iae'])}")
print(f"  ISE (∫e² dt)                      : {fmt(metrics['ise'])}")
print(f"  ITAE (∫t·|e| dt)                  : {fmt(metrics['itae'])}")
print(f"  Composite performance index       : {fmt(metrics['performance_index'])}")

## 8. Transition Zoom


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(tps, height_ref, "--", color="tab:gray", label=r"$-z_{ref}$")
ax.plot(tps, height_actual, color="tab:blue", label=r"$-z$")
ax.axhline(band_hi, color="tab:green", linestyle=":", linewidth=0.6, label="±5% band")
ax.axhline(band_lo, color="tab:green", linestyle=":", linewidth=0.6)
ax.axvline(T_STEP, color="tab:red", linestyle=":", linewidth=0.7, label="step")
if metrics['settling_time'] is not None:
    t_settle = T_STEP - 1.0 + metrics['settling_time']
    ax.axvline(t_settle, color="tab:orange", linestyle="-.", linewidth=0.8,
               label=f"settling = {metrics['settling_time']:.2f} s")
if metrics['rise_time'] is not None:
    t_rise = T_STEP - 1.0 + metrics['rise_time']
    ax.axvline(t_rise, color="tab:purple", linestyle="-.", linewidth=0.8,
               label=f"rise = {metrics['rise_time']:.2f} s")
ax.set_xlim(T_STEP - 1.0, min(T_END, T_STEP + 16.0))
ax.set_xlabel("time, s"); ax.set_ylabel("altitude, m")
ax.set_title("Transient response: zoom on the post-step window")
ax.grid(True); ax.legend(loc="best")
plt.tight_layout(); plt.show()

## 9. Conclusions

* The IHDP actor and critic now use two tracking states (`e_z`, `w_b`), so the transient is both fast and well damped.
* **Composite performance index ≈ 15** instead of ≈1150 in the single-state configuration.
* **Rise time ≈0.5 s, settling time ≈0.6 s**, with nearly zero static error.
* The control remains IHDP: there is no PD baseline, no step feedforward, and no reference look-ahead; `ΔT` is generated by the IHDP actor and constrained by the physical thrust range of the model.
